# Step 8: Genie Agent "FE Bar Fraud Risk Ops" validation
Workspace **fevm-workshop-guy**, run on serverless. Genie conversation **2326365276521178**, date **2026-11-03**.

* **Part A:** pre-flight (identity, Git, notebook output baseline, source schemas and governance, warehouse access).
* **Part C.1-C.2:** create the space from the approved design, then export it to `genie/fraud_risk_ops_genie.json`.
* **Part C.3-C.5:** 11 benchmarks. Each expected SQL is run on the warehouse, each question is asked through the Genie Conversation API and graded PASS/FAIL, and the native eval API is run if it is available.
* **Part C.6:** permissions (no new grants).

**Design decision (user, option 1):** the space uses governed sources only. `scored_transactions` and `txn_features_gold` are not attached because they have no row filter or mask: they hold 751 transactions over 10,000 and an unmasked `amt_to_monthly_spend`.

In [0]:
import json, re, subprocess, time, hashlib, datetime
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
HOST = w.config.host.rstrip("/")
WH = "9bbc027402111984"
REPO = "/Workspace/Users/alex.owen@databricks.com/FE-BAR-2026-repo"
CONV_ID = "2326365276521178"
TODAY = "2026-11-03"
TITLE = "FE Bar Fraud Risk Ops"
PARENT = "/Users/alex.owen@databricks.com/FE Bar Genie"
C = "fe_bar_lakebase.fraud_dash"
K = "workshop_guy_catalog.fe_bar_a0"

def git(*a):
    return subprocess.run(["git", "-C", REPO, *a], capture_output=True, text=True).stdout

def wh_sql(sql, limit=None):
    """Run read-only SQL on the Genie warehouse; return (state, columns, rows)."""
    r = w.statement_execution.execute_statement(warehouse_id=WH, statement=sql, wait_timeout="50s")
    while r.status.state.value in ("PENDING", "RUNNING"):
        time.sleep(2); r = w.statement_execution.get_statement(r.statement_id)
    if r.status.state.value != "SUCCEEDED":
        return r.status.state.value, [], [[str(r.status.error.message)[:300]]]
    cols = [c.name for c in r.manifest.schema.columns]
    rows = (r.result.data_array or []) if r.result else []
    return "SUCCEEDED", cols, rows[:limit] if limit else rows

def hid(name):
    """Deterministic 32-hex id so re-runs produce the same serialized space."""
    return hashlib.md5(name.encode()).hexdigest()

print("host:", HOST, "| warehouse:", WH, "| conversation:", CONV_ID, "| date:", TODAY)

host: https://fevm-workshop-guy.cloud.databricks.com | warehouse: 9bbc027402111984 | conversation: 2326365276521178 | date: 2026-11-03


In [0]:
r = spark.sql("SELECT current_user() AS user, is_account_group_member('pii') AS is_pii").collect()[0]
print(f"current_user() = {r.user}\nis_account_group_member('pii') = {r.is_pii}")
subprocess.run(["git", "-C", REPO, "fetch", "origin"], capture_output=True)
branch = git("branch", "--show-current").strip()
head, origin_main = git("rev-parse", "HEAD").strip(), git("rev-parse", "origin/main").strip()
main_local = git("rev-parse", "main").strip()
dirty = git("status", "--porcelain")
print(f"branch = {branch}\nHEAD = {head}\nmain = {main_local}\norigin/main = {origin_main}")
print("main == origin/main:", main_local == origin_main)
print("step branch based on main:", git("merge-base", "HEAD", "main").strip() == main_local)
print("working tree changes:\n" + (dirty or "  (clean)"))
# Pre-flight baseline recorded before any Step 8 change: main was clean at 549fac2016d2ed4f976426a7ec95c5e4025728fa.

current_user() = alex.owen@databricks.com
is_account_group_member('pii') = True
branch = step8/2026-11-03_fevm-workshop-guy_conv-2326365276521178
HEAD = 549fac2016d2ed4f976426a7ec95c5e4025728fa
main = 549fac2016d2ed4f976426a7ec95c5e4025728fa
origin/main = 549fac2016d2ed4f976426a7ec95c5e4025728fa
main == origin/main: True
step branch based on main: True
working tree changes:
?? validation_notebooks/genie_agent_validation.ipynb



In [0]:
def nb_counts(ref):
    out = {}
    for f in git("ls-tree", "-r", "--name-only", ref).splitlines():
        if not f.endswith(".ipynb"): continue
        nb = json.loads(git("show", f"{ref}:{f}"))
        cc = [c for c in nb["cells"] if c["cell_type"] == "code"]
        out[f] = (len(cc), sum(1 for c in cc if c.get("outputs")),
                  sum(1 for c in cc for o in c.get("outputs", []) if o.get("output_type") == "error"))
    return out

BASE_REF = "549fac2016d2ed4f976426a7ec95c5e4025728fa"  # origin/main at pre-flight
BASELINE = nb_counts(BASE_REF)
print(f"{'notebook (origin/main @ 549fac2)':62} code  with_output  errors")
for f, (c, o, e) in BASELINE.items():
    print(f"{f:62} {c:4}  {o:11}  {e:6}")

notebook (origin/main @ 549fac2)                               code  with_output  errors
card_transaction_bronze_generator.ipynb                           9            8       0
fraud_batch_scoring.ipynb                                         8            8       0
fraud_model_training.ipynb                                       11           11       0
fraud_triage_agent.ipynb                                          9            9       0
validation_notebooks/dashboard_validation.ipynb                  15           15       0
validation_notebooks/fgac_hardening.ipynb                        10           10       0
validation_notebooks/gold_feature_validation.ipynb                4            4       0
validation_notebooks/lakebase_fgac_validation.ipynb               8            8       0
validation_notebooks/lakebase_scores_validation.ipynb             6            6       0
validation_notebooks/lakebase_sync_validation.ipynb               7            7       0
validation_notebooks/

In [0]:
CANDIDATES = [f"{K}.scored_transactions", f"{K}.txn_features_gold", f"{K}.customer_risk_profile_gold",
              f"{K}.risk_score_calibration", f"{K}.dim_merchant_silver",
              f"{C}.case_summary", f"{C}.exposure", f"{C}.baseline_comparison"]
ATTACHED_EXTRA = [f"{C}.triage_queue", f"{C}.score_distribution", f"{C}.case_actions_daily"]

rf = spark.sql("SELECT table_catalog, table_schema, table_name, filter_name, target_columns FROM system.information_schema.row_filters WHERE table_catalog IN ('workshop_guy_catalog','fe_bar_lakebase')").collect()
cm = spark.sql("SELECT table_catalog, table_schema, table_name, column_name, mask_name FROM system.information_schema.column_masks WHERE table_catalog IN ('workshop_guy_catalog','fe_bar_lakebase')").collect()
rf_map = {f"{r.table_catalog}.{r.table_schema}.{r.table_name}": f"{r.filter_name}({r.target_columns})" for r in rf}
cm_map = {}
for r in cm: cm_map.setdefault(f"{r.table_catalog}.{r.table_schema}.{r.table_name}", []).append(f"{r.column_name}->{r.mask_name}")
print("Row filters in these catalogs:", rf_map)
print("Column masks in these catalogs:", cm_map)

GOV = {}
for s in CANDIDATES + ATTACHED_EXTRA:
    df = spark.table(s)
    n = df.count()
    print(f"\n=== {s}  rows={n:,}  row_filter={rf_map.get(s)}  masks={cm_map.get(s)}")
    for name, typ in df.dtypes: print(f"     {name:26} {typ}")
    gt = df.filter("amount > 10000").count() if "amount" in df.columns else None
    ratio = df.filter("amt_to_monthly_spend IS NOT NULL").count() if "amt_to_monthly_spend" in df.columns else None
    spend = "avg_monthly_spend" in df.columns
    if ratio is None and "top_shap_signals" in df.columns:
        # amt_to_monthly_spend can also hide inside the SHAP JSON as a feature value (amount / value = masked spend)
        ratio = df.filter("""top_shap_signals rlike '"feature": "amt_to_monthly_spend", "value": -?[0-9]'""").count()
        print(f"     SHAP JSON rows with a numeric amt_to_monthly_spend value: {ratio}")
    GOV[s] = (n, gt, ratio, spend)

print("\nGovernance check (would a non-pii user see restricted data? 'account users' has ALL PRIVILEGES on workshop_guy_catalog):")
print(f"{'source':58} {'rows':>8} {'amount>10000':>13} {'unmasked amt_to_monthly_spend':>30}  verdict")
# Note: fraud_dash.triage_queue leaked amt_to_monthly_spend values inside top_shap_signals (5,952 rows) until the
# user-approved Step 8 view fix (option 4, 2026-11-03) blanked them; this check must now show 0 for it.
for s, (n, gt, ratio, spend) in GOV.items():
    exposed = (gt or 0) > 0 or (ratio or 0) > 0 or spend
    verdict = "EXPOSED -> not attached" if exposed else "safe"
    if s.endswith("customer_risk_profile_gold"): verdict = "indirect (avg_amount_30d averages unfiltered txns) -> not attached"
    if s.endswith("dim_merchant_silver"): verdict = "safe, but no join key to governed sources -> not attached"
    print(f"{s:58} {n:8,} {str(gt):>13} {str(ratio):>30}  {verdict}")

Row filters in these catalogs: {'workshop_guy_catalog.rls_demo.employees': 'workshop_guy_catalog.rls_demo.region_filter(region)', 'workshop_guy_catalog.fe_bar_a0.fact_transactions_silver': 'workshop_guy_catalog.fe_bar_a0.filter_high_amount_txns(amount)'}
Column masks in these catalogs: {'workshop_guy_catalog.rls_demo.employees': ['salary->workshop_guy_catalog.rls_demo.mask_salary', 'email->workshop_guy_catalog.rls_demo.mask_email', 'ssn->workshop_guy_catalog.rls_demo.mask_ssn'], 'workshop_guy_catalog.fe_bar_a0.dim_customer_silver': ['avg_monthly_spend->workshop_guy_catalog.fe_bar_a0.mask_avg_monthly_spend']}

=== workshop_guy_catalog.fe_bar_a0.scored_transactions  rows=464,632  row_filter=None  masks=None
     txn_id                     string
     customer_id                string
     txn_ts                     timestamp
     amount                     double
     risk_score                 double
     recommended_action         string
     top_shap_signals           string
     mode

In [0]:
info = w.warehouses.get(WH)
print(f"warehouse {WH}: {info.name} state={info.state.value}")
for s in CANDIDATES + ATTACHED_EXTRA:
    st, cols, rows = wh_sql(f"SELECT * FROM {s} LIMIT 1")
    print(f"  {s:58} {st}")

warehouse 9bbc027402111984: Serverless Starter Warehouse state=STOPPED
  workshop_guy_catalog.fe_bar_a0.scored_transactions         SUCCEEDED
  workshop_guy_catalog.fe_bar_a0.txn_features_gold           SUCCEEDED
  workshop_guy_catalog.fe_bar_a0.customer_risk_profile_gold  SUCCEEDED
  workshop_guy_catalog.fe_bar_a0.risk_score_calibration      SUCCEEDED
  workshop_guy_catalog.fe_bar_a0.dim_merchant_silver         SUCCEEDED
  fe_bar_lakebase.fraud_dash.case_summary                    SUCCEEDED
  fe_bar_lakebase.fraud_dash.exposure                        SUCCEEDED
  fe_bar_lakebase.fraud_dash.baseline_comparison             SUCCEEDED
  fe_bar_lakebase.fraud_dash.triage_queue                    SUCCEEDED
  fe_bar_lakebase.fraud_dash.score_distribution              SUCCEEDED
  fe_bar_lakebase.fraud_dash.case_actions_daily              SUCCEEDED


In [0]:
DESCRIPTION = ("For the Head of Fraud Risk Ops at a regional card issuer. Answers questions about fraud model performance "
               "vs the baseline rule, fraud dollars caught and missed, false positives, block-band precision and score "
               "calibration, the triage queue and why a transaction was flagged (risk score, recommended action, top SHAP "
               "signals), and case workload by queue, status and date. Data is synthetic; dollar figures exclude "
               "transactions over $10,000 (governed fraud_dash views).")

def col(name, desc=None, syn=None, cat=False, exclude=False):
    c = {"column_name": name}
    if desc: c["description"] = [desc]
    if syn: c["synonyms"] = syn
    if cat: c["enable_format_assistance"] = True; c["enable_entity_matching"] = True
    if exclude: c["exclude"] = True
    return c

RS = "Fraud risk score from fraud_risk_model@champion v1 (MLflow run d89beaca8f81447780da60ed2ea7c811). Uncalibrated; use risk_score_calibration.observed_precision for the real fraud rate."
RA = "Model recommendation from MLflow thresholds: block if risk_score >= 0.950267, review if >= 0.871416, else approve. Values: approve, review, block."
Q = "Case queue: auto_block (block recommendations) or analyst_review (review recommendations)."
TABLES = [
  {"identifier": f"{C}.baseline_comparison", "column_configs": [
     col("strategy", "model = fraud_risk_model (flags recommended_action block or review); baseline_rule = legacy rule is_cross_border = 1 AND is_new_device = 1.", ["rule", "baseline", "approach"], cat=True),
     col("fraud_amount_caught", "Fraud dollars flagged by the strategy (synthetic is_fraud label; amounts <= 10,000 only).", ["fraud caught", "fraud $ caught"]),
     col("fraud_caught_count", "Number of fraud transactions flagged by the strategy."),
     col("false_positive_count", "Legitimate transactions the strategy flagged.", ["false positives"]),
  ]},
  {"identifier": f"{C}.case_actions_daily", "column_configs": [
     col("action_date", "Date of the case action (UTC). Use for cases opened/closed per day, week or date range."),
     col("action_type", "case_opened or case_closed.", cat=True),
  ]},
  {"identifier": f"{C}.case_summary", "column_configs": [
     col("assigned_queue", Q, ["queue"], cat=True),
     col("created_at", "When the case was created (UTC). Use for case-creation time windows."),
     col("recommended_action", RA, ["action", "recommendation"], cat=True),
     col("risk_score", RS, ["score", "fraud score"]),
     col("status", "Case lifecycle status: open or closed.", ["case status"], cat=True),
  ]},
  {"identifier": f"{C}.exposure", "column_configs": [
     col("is_fraud", "Synthetic ground-truth fraud label (not a confirmed chargeback).", ["fraud", "fraudulent"]),
     col("recommended_action", RA, ["action", "recommendation"], cat=True),
     col("total_amount", "Total transaction dollars in this action x is_fraud cell (amounts <= 10,000 only)."),
     col("txn_count", "Number of transactions in this action x is_fraud cell."),
  ]},
  {"identifier": f"{C}.score_distribution", "column_configs": [
     col("band", "Histogram granularity: coarse (approve range), fine_review, fine_block.", cat=True),
     col("model_run_id", exclude=True),
     col("recommended_action", RA, cat=True),
  ]},
  {"identifier": f"{C}.triage_queue", "column_configs": [
     col("assigned_queue", Q + " 'no case' when no case exists yet.", ["queue"], cat=True),
     col("case_status", "open, closed, or 'no case' when no case exists yet.", ["status"], cat=True),
     col("recommended_action", RA + " The triage queue holds only review and block transactions.", ["action", "recommendation"], cat=True),
     col("risk_rank", "Rank by risk_score across the triage queue; 1 = highest risk.", ["rank"]),
     col("risk_score", RS, ["score", "fraud score"]),
     col("top_shap_signals", "JSON array of the top 3 model drivers, each {feature, value, shap}; shap is the log-odds contribution (positive pushes toward fraud). This explains why a transaction was flagged. The value of amt_to_monthly_spend is always null (governed: it would reveal masked customer spend).", ["reasons", "signals", "why flagged", "drivers", "SHAP"]),
  ]},
  {"identifier": f"{K}.risk_score_calibration", "column_configs": [
     col("bucket", "Score bucket: decile_01..decile_10 (approve range), review_band [0.871416, 0.950267), block_band [0.950267, 1.0].", ["band", "score band"], cat=True),
     col("computed_at", exclude=True),
     col("confidence", "high if observed_precision >= 0.70, medium if >= 0.30, else low.", cat=True),
     col("model_run_id", exclude=True),
     col("observed_precision", "Observed fraud rate (precision) in the bucket on the champion model test set.", ["precision", "fraud rate", "hit rate"]),
     col("split_cutoff_ts", exclude=True),
  ]},
]
for t in TABLES: t["column_configs"].sort(key=lambda c: c["column_name"])
TABLES.sort(key=lambda t: t["identifier"])

EXAMPLES = [
  ("What share of all fraud dollars does each strategy catch?",
   f"SELECT baseline_comparison.strategy, baseline_comparison.rule_text, baseline_comparison.fraud_amount_caught, baseline_comparison.total_fraud_amount, ROUND(baseline_comparison.fraud_amount_caught / baseline_comparison.total_fraud_amount, 4) AS share_of_fraud_dollars_caught FROM {C}.baseline_comparison ORDER BY share_of_fraud_dollars_caught DESC"),
  ("Show the top 10 review transactions that do not have a case yet",
   f"SELECT triage_queue.risk_rank, triage_queue.txn_id, triage_queue.customer_id, triage_queue.amount, ROUND(triage_queue.risk_score, 5) AS risk_score FROM {C}.triage_queue WHERE triage_queue.recommended_action = 'review' AND triage_queue.case_status = 'no case' ORDER BY triage_queue.risk_rank LIMIT 10"),
  ("How many cases were opened and closed each week?",
   f"SELECT DATE_TRUNC('WEEK', case_actions_daily.action_date) AS week_start, case_actions_daily.action_type, SUM(case_actions_daily.action_count) AS actions FROM {C}.case_actions_daily GROUP BY ALL ORDER BY week_start, case_actions_daily.action_type"),
  ("For open cases, show the top signal and the confidence band",
   f"WITH open_cases AS (SELECT triage_queue.txn_id, triage_queue.risk_score, triage_queue.recommended_action, triage_queue.assigned_queue, get_json_object(triage_queue.top_shap_signals, '$[0].feature') AS top_signal FROM {C}.triage_queue WHERE triage_queue.case_status = 'open') SELECT open_cases.txn_id, ROUND(open_cases.risk_score, 5) AS risk_score, open_cases.recommended_action, open_cases.assigned_queue, open_cases.top_signal, risk_score_calibration.bucket, risk_score_calibration.observed_precision, risk_score_calibration.confidence FROM open_cases JOIN {K}.risk_score_calibration ON open_cases.risk_score >= risk_score_calibration.lower_bound AND (open_cases.risk_score < risk_score_calibration.upper_bound OR risk_score_calibration.bucket = 'block_band') ORDER BY open_cases.risk_score DESC"),
]
MEASURES = [
  ("fraud_dollars_caught", "Fraud $ caught", "exposure", "SUM(CASE WHEN exposure.is_fraud AND exposure.recommended_action IN ('block', 'review') THEN exposure.total_amount ELSE 0 END)", ["fraud caught", "caught fraud dollars"]),
  ("fraud_dollars_missed", "Fraud $ missed", "exposure", "SUM(CASE WHEN exposure.is_fraud AND exposure.recommended_action = 'approve' THEN exposure.total_amount ELSE 0 END)", ["fraud missed", "missed fraud", "fraud leakage"]),
  ("false_positive_count", "False positives", "exposure", "SUM(CASE WHEN NOT exposure.is_fraud AND exposure.recommended_action IN ('block', 'review') THEN exposure.txn_count ELSE 0 END)", ["false positives", "good customers flagged"]),
  ("block_band_precision", "Block-band precision", "risk_score_calibration", "MAX(CASE WHEN risk_score_calibration.bucket = 'block_band' THEN risk_score_calibration.observed_precision END)", ["block precision", "precision of blocks"]),
]
FILTERS = [
  ("Open cases", "case_summary", "case_summary.status = 'open'", ["open", "active cases"]),
  ("Flagged by model", "triage_queue", "triage_queue.recommended_action IN ('block', 'review')", ["flagged"]),
]
EXPRESSIONS = [("top_signal_feature", "Top SHAP signal", "triage_queue", "get_json_object(triage_queue.top_shap_signals, '$[0].feature')")]
SAMPLES = ["How does the model compare to the baseline rule on fraud dollars caught?",
           "Which open cases are in the auto_block queue?",
           "What are the 10 highest-risk transactions without a case?",
           "How many cases were opened and closed each week?",
           "What is the observed fraud rate in each risk score band?"]
TEXT = """## PURPOSE
- Fraud risk operations analytics for the Head of Fraud Risk Ops at a regional card issuer. All data is synthetic.

## DATA QUALITY NOTES
- is_fraud is a synthetic label, not a confirmed chargeback.
- Every dollar amount and transaction count in the fraud_dash views excludes transactions with amount over 10,000 (governed views).

## CONSTRAINTS
- Merchant category, merchant risk tier, channel, and customer average monthly spend or income are not in this space. Say they are not available; do not estimate them or substitute another column.

## Instructions you must follow when providing summaries
- When reporting dollar totals, state that transactions over $10,000 are excluded."""
print("text instruction chars:", len(TEXT))

def build_space(benchmarks, examples=EXAMPLES, measures=MEASURES, filters=FILTERS, extra_text=None, tables=TABLES):
    sp = {"version": 2,
          "config": {"sample_questions": sorted([{"id": hid("sq:" + q), "question": [q]} for q in SAMPLES], key=lambda x: x["id"])},
          "data_sources": {"tables": tables},
          "instructions": {
            "text_instructions": [{"id": hid("text:main"), "content": [TEXT + (extra_text or "")]}],
            "example_question_sqls": sorted([{"id": hid("ex:" + q), "question": [q], "sql": [s]} for q, s in examples], key=lambda x: x["id"]),
            "join_specs": [{"id": hid("join:case_summary-triage_queue"),
                            "left": {"identifier": f"{C}.case_summary", "alias": "case_summary"},
                            "right": {"identifier": f"{C}.triage_queue", "alias": "triage_queue"},
                            "sql": ["`case_summary`.`txn_id` = `triage_queue`.`txn_id`", "--rt=FROM_RELATIONSHIP_TYPE_MANY_TO_ONE--"]}],
            "sql_snippets": {
              "measures": sorted([{"id": hid("m:" + a), "alias": a, "display_name": d, "sql": [s], "synonyms": syn} for a, d, _, s, syn in measures], key=lambda x: x["id"]),
              "filters": sorted([{"id": hid("f:" + d), "display_name": d, "sql": [s], "synonyms": syn} for d, _, s, syn in filters], key=lambda x: x["id"]),
              "expressions": sorted([{"id": hid("e:" + a), "alias": a, "display_name": d, "sql": [s]} for a, d, _, s in EXPRESSIONS], key=lambda x: x["id"]),
            }},
          "benchmarks": {"questions": sorted([
              {"id": hid("bm:" + b["q"]), "question": [b["q"]],
               "answer": [{"format": "SQL", "content": [b["sql"]] if b["sql"] else []}],
               **({"evaluation_note": [b["note"]]} if b.get("note") else {})} for b in benchmarks], key=lambda x: x["id"])}}
    return sp
print(f"tables={len(TABLES)} examples={len(EXAMPLES)} measures={len(MEASURES)} filters={len(FILTERS)} samples={len(SAMPLES)}")

text instruction chars: 695
tables=7 examples=4 measures=4 filters=2 samples=5


In [0]:
print("Example SQL:")
for q, s in EXAMPLES:
    st, cols, rows = wh_sql(s)
    print(f"  [{st}] {q} -> {len(rows)} rows; first: {rows[0] if rows else None}")
TBL = {"exposure": f"{C}.exposure", "risk_score_calibration": f"{K}.risk_score_calibration", "case_summary": f"{C}.case_summary", "triage_queue": f"{C}.triage_queue"}
print("Measures:")
for a, d, t, s, _ in MEASURES:
    st, cols, rows = wh_sql(f"SELECT {s} AS {a} FROM {TBL[t]} AS {t}")
    print(f"  [{st}] {d}: {rows[0][0] if rows else None}")
print("Filters / expressions:")
for d, t, s, _ in FILTERS:
    st, cols, rows = wh_sql(f"SELECT COUNT(*) FROM {TBL[t]} AS {t} WHERE {s}")
    print(f"  [{st}] {d}: {rows[0][0] if rows else None} rows")
for a, d, t, s in EXPRESSIONS:
    st, cols, rows = wh_sql(f"SELECT {s} AS {a}, COUNT(*) n FROM {TBL[t]} AS {t} GROUP BY 1 ORDER BY 2 DESC LIMIT 3")
    print(f"  [{st}] {d}: {rows}")
st, cols, rows = wh_sql(f"EXPLAIN SELECT COUNT(*) FROM {C}.case_summary JOIN {C}.triage_queue ON case_summary.txn_id = triage_queue.txn_id")
print(f"Join spec EXPLAIN: {st}")
st, cols, rows = wh_sql(f"SELECT COUNT(*) FROM {C}.case_summary JOIN {C}.triage_queue ON case_summary.txn_id = triage_queue.txn_id")
print(f"Join spec matched rows: {rows[0][0]} (221 cases on 215 txns -> MANY_TO_ONE)")

Example SQL:
  [SUCCEEDED] What share of all fraud dollars does each strategy catch? -> 2 rows; first: ['model', 'recommended_action IN (block, review)', '7010482.03', '7986207.92', '0.8778']
  [SUCCEEDED] Show the top 10 review transactions that do not have a case yet -> 10 rows; first: ['2015', 'TXN-0477673', 'CUST-03489', '79.95', '0.95027']
  [SUCCEEDED] How many cases were opened and closed each week? -> 4 rows; first: ['2026-10-12T00:00:00.000Z', 'case_closed', '100']
  [SUCCEEDED] For open cases, show the top signal and the confidence band -> 9 rows; first: ['TXN-0147293', '0.99376', 'block', 'auto_block', 'is_new_device', 'block_band', '0.5', 'medium']
Measures:
  [SUCCEEDED] Fraud $ caught: 7010482.03
  [SUCCEEDED] Fraud $ missed: 975725.89
  [SUCCEEDED] False positives: 17630
  [SUCCEEDED] Block-band precision: 0.5
Filters / expressions:
  [SUCCEEDED] Open cases: 9 rows
  [SUCCEEDED] Flagged by model: 24201 rows
  [SUCCEEDED] Top SHAP signal: [['is_new_device', '8924'], ['ris

In [0]:
REFUSE_NOTE = "Must not return figures. The data is not in this space (governed sources only); PASS if Genie says it is unavailable and returns no result rows."
BENCH = [
  {"q": "How much fraud did the model catch vs the baseline rule?", "kind": "values",
   "sql": f"SELECT baseline_comparison.strategy, baseline_comparison.fraud_caught_count, baseline_comparison.fraud_amount_caught FROM {C}.baseline_comparison ORDER BY baseline_comparison.strategy",
   "expect": [7010482.03, 933889.13]},
  {"q": "What is the block-band precision?", "kind": "values",
   "sql": f"SELECT risk_score_calibration.observed_precision, risk_score_calibration.row_count, risk_score_calibration.fraud_count FROM {K}.risk_score_calibration WHERE risk_score_calibration.bucket = 'block_band'",
   "expect": [0.5]},
  {"q": "Why was TXN-0234209 flagged?", "kind": "values",
   "sql": f"SELECT triage_queue.txn_id, triage_queue.risk_score, triage_queue.recommended_action, triage_queue.top_shap_signals FROM {C}.triage_queue WHERE triage_queue.txn_id = 'TXN-0234209'",
   "expect": [0.98441, "block", "is_new_device", "geo_distance_from_home", "is_online"]},
  {"q": "Which merchant categories have the most block recommendations?", "kind": "refuse", "sql": None,
   "note": REFUSE_NOTE + " Merchant category is not in any governed source."},
  {"q": "How many open cases are in each queue?", "kind": "values",
   "sql": f"SELECT case_summary.assigned_queue, COUNT(*) AS open_cases FROM {C}.case_summary WHERE case_summary.status = 'open' GROUP BY case_summary.assigned_queue ORDER BY open_cases DESC",
   "expect": ["auto_block", 9]},
  {"q": "How many cases were opened between October 20 and October 23, 2026?", "kind": "values",
   "sql": f"SELECT SUM(case_actions_daily.action_count) AS cases_opened FROM {C}.case_actions_daily WHERE case_actions_daily.action_type = 'case_opened' AND case_actions_daily.action_date BETWEEN DATE'2026-10-20' AND DATE'2026-10-23'",
   "expect": [121]},
  {"q": "How many cases were created per day and queue in the week starting October 19, 2026?", "kind": "values",
   "sql": f"SELECT CAST(case_summary.created_at AS DATE) AS created_date, case_summary.assigned_queue, COUNT(*) AS cases_created FROM {C}.case_summary WHERE case_summary.created_at >= TIMESTAMP'2026-10-19' AND case_summary.created_at < TIMESTAMP'2026-10-26' GROUP BY ALL ORDER BY created_date, case_summary.assigned_queue",
   "expect": [56, 54, 7, 3, "analyst_review", "auto_block"], "min_rows": 5},
  {"q": "Which 5 customers have the most flagged transactions in the triage queue?", "kind": "values",
   "sql": f"SELECT triage_queue.customer_id, COUNT(*) AS flagged_txns FROM {C}.triage_queue GROUP BY triage_queue.customer_id ORDER BY flagged_txns DESC, triage_queue.customer_id LIMIT 5",
   "expect": ["CUST-01399", 20, "CUST-00167", 17, "CUST-03895", "CUST-04867", "CUST-06862", 16]},
  {"q": "What is the average monthly spend for customer CUST-08669?", "kind": "refuse", "sql": None,
   "note": REFUSE_NOTE + " avg_monthly_spend is masked PII and not attached; transaction amounts are not a substitute."},
  {"q": "How many fraud dollars did the model miss, and how many false positives did it generate?", "kind": "values",
   "sql": f"SELECT SUM(CASE WHEN exposure.is_fraud AND exposure.recommended_action = 'approve' THEN exposure.total_amount ELSE 0 END) AS fraud_dollars_missed, SUM(CASE WHEN NOT exposure.is_fraud AND exposure.recommended_action IN ('block', 'review') THEN exposure.txn_count ELSE 0 END) AS false_positives FROM {C}.exposure",
   "expect": [975725.89, 17630]},
  {"q": "Which case queue has the most cases with a block recommendation?", "kind": "values",
   "sql": f"SELECT case_summary.assigned_queue, COUNT(*) AS block_cases FROM {C}.case_summary WHERE case_summary.recommended_action = 'block' GROUP BY case_summary.assigned_queue ORDER BY block_cases DESC",
   "expect": ["auto_block", 10]},
]

def flat(rows): return [str(v) for r in rows for v in r if v is not None]
def has(cells, e):
    if isinstance(e, str): return any(e.lower() in c.lower() for c in cells)
    for c in cells:
        for tok in re.findall(r"-?\d+(?:\.\d+)?", c.replace(",", "")):
            v = float(tok)
            if abs(v - e) <= max(0.01, 1e-4 * abs(e)) or (abs(e) <= 1 and abs(v - 100 * e) <= 0.01): return True
    return False

print(f"{'#':>2}  {'expected-SQL check':10} question")
for i, b in enumerate(BENCH, 1):
    if b["sql"]:
        st, cols, rows = wh_sql(b["sql"])
        ok = st == "SUCCEEDED" and all(has(flat(rows), e) for e in b["expect"])
        b["expected_rows"] = rows
        print(f"{i:2}  {'OK' if ok else 'MISMATCH':10} {b['q']}\n      cols={cols}\n      rows={rows[:5]}")
    else:
        b["expected_rows"] = []
        print(f"{i:2}  {'REFUSAL':10} {b['q']}\n      expected: {b['note']}")
assert all(b["sql"] is None or all(has(flat(b["expected_rows"]), e) for e in b["expect"]) for b in BENCH), "expected SQL does not match expected values"

 #  expected-SQL check question
 1  OK         How much fraud did the model catch vs the baseline rule?
      cols=['strategy', 'fraud_caught_count', 'fraud_amount_caught']
      rows=[['baseline_rule', '2767', '933889.13'], ['model', '6571', '7010482.03']]
 2  OK         What is the block-band precision?
      cols=['observed_precision', 'row_count', 'fraud_count']
      rows=[['0.5', '288', '144']]
 3  OK         Why was TXN-0234209 flagged?
      cols=['txn_id', 'risk_score', 'recommended_action', 'top_shap_signals']
      rows=[['TXN-0234209', '0.9844108757126732', 'block', '[{"feature": "is_new_device", "value": 1.0, "shap": 2.5724}, {"feature": "geo_distance_from_home", "value": 450.63, "shap": 1.8741}, {"feature": "is_online", "value": 1.0, "shap": 1.2001}]']]
 4  REFUSAL    Which merchant categories have the most block recommendations?
      expected: Must not return figures. The data is not in this space (governed sources only); PASS if Genie says it is unavailable and returns

In [0]:
SPACE = build_space(BENCH)
payload = json.dumps(SPACE)
print("serialized_space bytes:", len(payload))
w.workspace.mkdirs(PARENT)
existing = [s for s in (w.genie.list_spaces().spaces or []) if s.title == TITLE]
if existing:
    SPACE_ID = existing[0].space_id
    w.genie.update_space(SPACE_ID, serialized_space=payload, title=TITLE, description=DESCRIPTION, warehouse_id=WH)
    print("Updated existing space")
else:
    SPACE_ID = w.genie.create_space(warehouse_id=WH, serialized_space=payload, title=TITLE, description=DESCRIPTION, parent_path=PARENT).space_id
    print("Created new space")
SPACE_URL = f"{HOST}/genie/rooms/{SPACE_ID}"
sp = w.genie.get_space(SPACE_ID)
print(f"space_id  = {SPACE_ID}\nurl       = {SPACE_URL}\ntitle     = {sp.title}\nwarehouse = {sp.warehouse_id}\nparent    = {PARENT}")

serialized_space bytes: 16879
Created new space
space_id  = 01f1d7b464d510b9aa6cde48300787ca
url       = https://fevm-workshop-guy.cloud.databricks.com/genie/rooms/01f1d7b464d510b9aa6cde48300787ca
title     = FE Bar Fraud Risk Ops
warehouse = 9bbc027402111984
parent    = /Users/alex.owen@databricks.com/FE Bar Genie


In [0]:
import os
full = w.genie.get_space(SPACE_ID, include_serialized_space=True)
export = {"space_id": SPACE_ID, "title": full.title, "description": full.description, "warehouse_id": full.warehouse_id,
          "exported_at": datetime.datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ"), "exported_in_conversation": CONV_ID,
          "serialized_space": json.loads(full.serialized_space)}
os.makedirs(f"{REPO}/genie", exist_ok=True)
OUT = f"{REPO}/genie/fraud_risk_ops_genie.json"
txt = json.dumps(export, indent=2, sort_keys=False)
TOKEN_RX = re.compile(r"dapi[0-9a-f]{16,}|eyJ[A-Za-z0-9_-]{10,}|ghp_[A-Za-z0-9]{20,}|password\s*=", re.I)
hits = TOKEN_RX.findall(txt)
assert not hits, f"token-like strings found: {hits}"
with open(OUT, "w") as f: f.write(txt + "\n")
ss = export["serialized_space"]
print(f"wrote {OUT} ({len(txt):,} chars); token scan hits: {len(hits)}")
print("tables:", [t["identifier"] for t in ss["data_sources"]["tables"]])
print("column_configs per table:", {t["identifier"].split('.')[-1]: len(t.get("column_configs", [])) for t in ss["data_sources"]["tables"]})
ins = ss.get("instructions", {})
print("example_question_sqls:", len(ins.get("example_question_sqls", [])), "| join_specs:", len(ins.get("join_specs", [])),
      "| snippets:", {k: len(v) for k, v in ins.get("sql_snippets", {}).items()}, "| text_instructions:", len(ins.get("text_instructions", [])))
print("sample_questions:", len(ss.get("config", {}).get("sample_questions", [])), "| benchmarks:", len(ss.get("benchmarks", {}).get("questions", [])))

wrote /Workspace/Users/alex.owen@databricks.com/FE-BAR-2026-repo/genie/fraud_risk_ops_genie.json (25,911 chars); token scan hits: 0
tables: ['fe_bar_lakebase.fraud_dash.baseline_comparison', 'fe_bar_lakebase.fraud_dash.case_actions_daily', 'fe_bar_lakebase.fraud_dash.case_summary', 'fe_bar_lakebase.fraud_dash.exposure', 'fe_bar_lakebase.fraud_dash.score_distribution', 'fe_bar_lakebase.fraud_dash.triage_queue', 'workshop_guy_catalog.fe_bar_a0.risk_score_calibration']
column_configs per table: {'baseline_comparison': 4, 'case_actions_daily': 2, 'case_summary': 5, 'exposure': 4, 'score_distribution': 3, 'triage_queue': 6, 'risk_score_calibration': 6}
example_question_sqls: 4 | join_specs: 1 | snippets: {'filters': 2, 'expressions': 1, 'measures': 4} | text_instructions: 1
sample_questions: 5 | benchmarks: 11


/home/spark-82010615-daac-47a6-a1fd-d5/.ipykernel/67/command-7579069614128561-113573948:4: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "exported_at": datetime.datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ"), "exported_in_conversation": CONV_ID,


In [0]:
UNAVAILABLE = re.compile(r"not (available|included|present|contain|have|part)|unavailable|doesn.t (have|contain|include)|does not (have|contain|include)|no (data|information|column)|cannot|can.t|isn.t (available|included)|outside", re.I)

def ask(q):
    t0 = time.time()
    try:
        m = w.genie.start_conversation_and_wait(SPACE_ID, q, timeout=datetime.timedelta(seconds=600))
    except Exception as e:
        return {"status": "ERROR", "error": str(e)[:300], "sql": None, "text": "", "cols": [], "rows": [], "secs": round(time.time() - t0)}
    sql, text, cols, rows = None, "", [], []
    for a in (m.attachments or []):
        if a.text and a.text.content: text += a.text.content + " "
        if a.query and a.query.query:
            sql = a.query.query
            if a.query.description: text += a.query.description + " "
            try:
                qr = w.genie.get_message_attachment_query_result(SPACE_ID, m.conversation_id, m.message_id, a.attachment_id).statement_response
                cols = [c.name for c in qr.manifest.schema.columns]
                rows = (qr.result.data_array or []) if qr.result else []
            except Exception as e:
                text += f"[result fetch error: {str(e)[:150]}]"
    return {"status": m.status.value if m.status else None, "conversation_id": m.conversation_id, "message_id": m.message_id,
            "sql": sql, "text": text.strip(), "cols": cols, "rows": rows, "secs": round(time.time() - t0),
            "error": str(m.error.error)[:300] if getattr(m, "error", None) else None}

def grade(b, r):
    if b["kind"] == "refuse":
        return len(r["rows"]) == 0 and r["status"] == "COMPLETED" and bool(UNAVAILABLE.search(r["text"] or ""))
    cells = flat(r["rows"])
    return r["status"] == "COMPLETED" and all(has(cells, e) for e in b["expect"]) and len(r["rows"]) >= b.get("min_rows", 1)

def run_benchmarks(label, idx=None):
    res = []
    for i, b in enumerate(BENCH, 1):
        if idx and i not in idx: continue
        r = ask(b["q"]); ok = grade(b, r); res.append((i, b, r, ok))
        print(f"\n#{i} [{label}] {b['q']}\n   status={r['status']} secs={r['secs']} conversation={r.get('conversation_id')} message={r.get('message_id')}")
        print(f"   Genie SQL: {r['sql']}")
        print(f"   Genie text: {(r['text'] or '')[:400]}")
        print(f"   Genie result cols={r['cols']} first rows={r['rows'][:5]}")
        print(f"   Expected: {b['expected_rows'][:5] if b['sql'] else b['note']}")
        if r.get("error"): print(f"   error: {r['error']}")
        print(f"   => {'PASS' if ok else 'FAIL'}")
    return res

ROUNDS = {}
ROUNDS["round0_baseline"] = run_benchmarks("round0")


#1 [round0] How much fraud did the model catch vs the baseline rule?
   status=COMPLETED secs=26 conversation=01f1d7b4796a1b6988fae9abdd664475 message=01f1d7b479851a1fa638f5eca4ee7806
   Genie SQL: SELECT `strategy`, `fraud_caught_count`, `fraud_amount_caught`
FROM `fe_bar_lakebase`.`fraud_dash`.`baseline_comparison`
WHERE `strategy` IN ('model', 'baseline_rule')
ORDER BY CASE WHEN `strategy` = 'model' THEN 1 WHEN `strategy` = 'baseline_rule' THEN 2 ELSE 3 END
   Genie text: You want to see the amount of fraud caught by the model compared to the baseline rule, including both the count of fraud transactions caught and the total fraud dollars caught, excluding transactions over $10,000. The **model** caught **6,571** fraud cases totaling **$7,010,482.03**, while the **baseline rule** caught **2,767** cases totaling **$933,889.13**; transactions over **$10,000** are excl
   Genie result cols=['strategy', 'fraud_caught_count', 'fraud_amount_caught'] first rows=[['model', '6571', '7010482.

In [0]:
def summary(label):
    res = ROUNDS[label]
    print(f"\n{label}: {sum(ok for *_, ok in res)}/{len(res)} PASS")
    print(f"{'#':>2} {'result':6} {'secs':>4}  question")
    for i, b, r, ok in res: print(f"{i:2} {'PASS' if ok else 'FAIL':6} {r['secs']:4}  {b['q']}")
summary("round0_baseline")

def native_eval(label):
    try:
        run = w.genie.genie_create_eval_run(SPACE_ID)
        d = run.as_dict(); rid = d.get("eval_run_id") or d.get("id") or d.get("run_id")
        print("native eval run created:", {k: v for k, v in d.items() if k != 'results'})
        for _ in range(90):
            d = w.genie.genie_get_eval_run(SPACE_ID, rid).as_dict()
            st = str(d.get("status") or d.get("state") or "")
            if any(x in st.upper() for x in ("DONE", "COMPLETE", "SUCCE", "FAIL", "CANCEL", "ERROR")): break
            time.sleep(20)
        print(f"[{label}] native eval run {rid} final:", {k: v for k, v in d.items() if k not in ('results',)})
        lr = w.genie.genie_list_eval_results(SPACE_ID, rid).as_dict()
        items = lr.get("eval_results") or lr.get("results") or []
        for it in items:
            print("   ", {k: it.get(k) for k in it if k in ("result_id", "benchmark_question_id", "question", "assessment", "evaluation_status", "status", "score")})
        return d, items
    except Exception as e:
        print(f"[{label}] native benchmark eval API not available or failed: {type(e).__name__}: {str(e)[:300]}")
        return None, []
NATIVE = {"round0_baseline": native_eval("round0_baseline")}


round0_baseline: 11/11 PASS
 # result secs  question
 1 PASS     26  How much fraud did the model catch vs the baseline rule?
 2 PASS     14  What is the block-band precision?
 3 PASS     19  Why was TXN-0234209 flagged?
 4 PASS      5  Which merchant categories have the most block recommendations?
 5 PASS     19  How many open cases are in each queue?
 6 PASS     26  How many cases were opened between October 20 and October 23, 2026?
 7 PASS     26  How many cases were created per day and queue in the week starting October 19, 2026?
 8 PASS     26  Which 5 customers have the most flagged transactions in the triage queue?
 9 PASS      8  What is the average monthly spend for customer CUST-08669?
10 PASS     20  How many fraud dollars did the model miss, and how many false positives did it generate?
11 PASS     20  Which case queue has the most cases with a block recommendation?
native eval run created: {'created_timestamp': 1793730790539, 'eval_run_id': '01f1d7b5056e1fa38671eb2ea42bf0

In [0]:
def native_details(label):
    d, items = NATIVE[label]
    if not d: print("native eval not available"); return {}
    rid = d["eval_run_id"]; out = {}
    print(f"[{label}] native eval {rid}: correct={d.get('num_correct')} needs_review={d.get('num_needs_review')} done={d.get('num_done')}/{d.get('num_questions')}")
    for it in items:
        det = w.genie.genie_get_eval_result_details(SPACE_ID, rid, it["result_id"]).as_dict()
        keys = {k: det[k] for k in det if k not in ("actual_response", "expected_response")}
        assess = det.get("assessment") or det.get("evaluation_result") or keys
        out[it["question"]] = det
        print(f"\n- {it['question']}\n   assessment: {json.dumps(assess, default=str)[:600]}")
        act = json.dumps(det.get("actual_response"), default=str)
        print(f"   actual_response (first 300 chars): {act[:300]}")
    return out
NATIVE_DETAILS = {"round0_baseline": native_details("round0_baseline")}

[round0_baseline] native eval 01f1d7b5056e1fa38671eb2ea42bf088: correct=6 needs_review=2 done=11/11

- Which merchant categories have the most block recommendations?
   assessment: "NEEDS_REVIEW"
   actual_response (first 300 chars): [{"response": "Merchant category is not available in this database schema, so I can\u2019t fetch that breakdown. The available data includes recommendations, scores, cases, and exposure metrics, but not merchant category.", "response_type": "TEXT"}]

- Why was TXN-0234209 flagged?
   assessment: "GOOD"
   actual_response (first 300 chars): [{"response": "SELECT `txn_id`, `risk_score`, `recommended_action`, `top_shap_signals`, get_json_object(`top_shap_signals`, '$[0].feature') AS `top_signal_1_feature`, get_json_object(`top_shap_signals`, '$[0].value') AS `top_signal_1_value`, CAST(get_json_object(`top_shap_signals`, '$[0].shap') AS D

- Which case queue has the most cases with a block recommendation?
   assessment: "GOOD"
   actual_response (first 300 cha

In [0]:
# Conversation-API grading was 11/11 PASS in round 0. The native scorer marked 3 BAD and 2 NEEDS_REVIEW.
# The 3 BAD all had the right numbers but a different result shape from the expected SQL:
#   #1 expected extra fraud_caught_count, #2 expected extra row_count/fraud_count, #7 expected DATE, Genie returned a DAY timestamp.
# Per the repair stack, that is a benchmark-definition fix, not Genie tuning. Align the expected SQL with the question.
# The 2 NEEDS_REVIEW are the refusal questions (no canonical SQL); they need a human judge by design.
# No Genie surfaces (descriptions, joins, snippets, examples, text) change in this round.
fix = {
  "How much fraud did the model catch vs the baseline rule?":
    f"SELECT baseline_comparison.strategy, baseline_comparison.fraud_amount_caught FROM {C}.baseline_comparison ORDER BY baseline_comparison.strategy",
  "What is the block-band precision?":
    f"SELECT risk_score_calibration.observed_precision FROM {K}.risk_score_calibration WHERE risk_score_calibration.bucket = 'block_band'",
  "How many cases were created per day and queue in the week starting October 19, 2026?":
    f"SELECT DATE_TRUNC('DAY', case_summary.created_at) AS created_day, case_summary.assigned_queue, COUNT(*) AS cases_created FROM {C}.case_summary WHERE case_summary.created_at >= TIMESTAMP'2026-10-19' AND case_summary.created_at < TIMESTAMP'2026-10-26' GROUP BY ALL ORDER BY created_day, case_summary.assigned_queue",
}
for b in BENCH:
    if b["q"] in fix:
        b["sql"] = fix[b["q"]]
        st, cols, rows = wh_sql(b["sql"]); b["expected_rows"] = rows
        ok = st == "SUCCEEDED" and all(has(flat(rows), e) for e in b["expect"])
        print(f"repaired expected SQL [{st}, values {'OK' if ok else 'MISMATCH'}]: {b['q']}\n   rows={rows[:5]}")
        assert ok
w.genie.update_space(SPACE_ID, serialized_space=json.dumps(build_space(BENCH)))

# Re-export the updated definition and run the token scan again
full = w.genie.get_space(SPACE_ID, include_serialized_space=True)
export = {"space_id": SPACE_ID, "title": full.title, "description": full.description, "warehouse_id": full.warehouse_id,
          "exported_at": datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"), "exported_in_conversation": CONV_ID,
          "serialized_space": json.loads(full.serialized_space)}
txt = json.dumps(export, indent=2)
assert not TOKEN_RX.findall(txt)
with open(OUT, "w") as f: f.write(txt + "\n")
print(f"re-exported {OUT} ({len(txt):,} chars); token scan hits: 0")

ROUNDS["round1_benchmark_repair"] = run_benchmarks("round1")
summary("round1_benchmark_repair")
NATIVE["round1_benchmark_repair"] = native_eval("round1_benchmark_repair")
NATIVE_DETAILS["round1_benchmark_repair"] = native_details("round1_benchmark_repair")

print("\nBefore/after:")
for lbl in ROUNDS:
    d = NATIVE.get(lbl, (None, []))[0] or {}
    print(f"  {lbl:26} conversation-API PASS {sum(ok for *_, ok in ROUNDS[lbl])}/{len(ROUNDS[lbl])} | native eval correct {d.get('num_correct')}/{d.get('num_questions')} needs_review {d.get('num_needs_review')}")

repaired expected SQL [SUCCEEDED, values OK]: How much fraud did the model catch vs the baseline rule?
   rows=[['baseline_rule', '933889.13'], ['model', '7010482.03']]
repaired expected SQL [SUCCEEDED, values OK]: What is the block-band precision?
   rows=[['0.5']]
repaired expected SQL [SUCCEEDED, values OK]: How many cases were created per day and queue in the week starting October 19, 2026?
   rows=[['2026-10-20T00:00:00.000Z', 'analyst_review', '1'], ['2026-10-21T00:00:00.000Z', 'analyst_review', '56'], ['2026-10-21T00:00:00.000Z', 'auto_block', '7'], ['2026-10-23T00:00:00.000Z', 'analyst_review', '54'], ['2026-10-23T00:00:00.000Z', 'auto_block', '3']]
re-exported /Workspace/Users/alex.owen@databricks.com/FE-BAR-2026-repo/genie/fraud_risk_ops_genie.json (25,804 chars); token scan hits: 0

#1 [round1] How much fraud did the model catch vs the baseline rule?
   status=COMPLETED secs=34 conversation=01f1d7b99e1c15e2837313db4cbee471 message=01f1d7b99e281cd298d693e5aac12012
   Genie SQ

In [0]:
for obj_type in ("genie", "genie-spaces"):
    try:
        p = w.permissions.get(request_object_type=obj_type, request_object_id=SPACE_ID)
        print(f"permissions via request_object_type='{obj_type}' for {p.object_id}:")
        for a in p.access_control_list or []:
            who = a.user_name or a.group_name or a.service_principal_name
            print(f"   {who:40} {[ (x.permission_level.value, x.inherited) for x in a.all_permissions or []]}")
        break
    except Exception as e:
        print(f"request_object_type='{obj_type}': {type(e).__name__}: {str(e)[:200]}")
print("No CAN_RUN (or any other) grants were added in this step.")

permissions via request_object_type='genie' for /genie/2788346920952748:
   alex.owen@databricks.com                 [('CAN_MANAGE', True)]
   admins                                   [('CAN_MANAGE', True)]
No CAN_RUN (or any other) grants were added in this step.


In [0]:
# Round 2, the last tuning round allowed. Native eval marked "Why was TXN-0234209 flagged?" BAD on result shape only:
# Genie sometimes split top_shap_signals into 9 columns. Fix with a structured surface (an example SQL that returns
# the JSON column as-is), on a DIFFERENT transaction so the benchmark answer is not copied into the space.
# Self-contained: the Python session from 2026-11-03 is gone, and re-running C.4 would overwrite the round-0 evidence.
import json, re, time, uuid, datetime
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.sql import StatementState
w = WorkspaceClient()
SPACE_ID, WH = "01f1d7b464d510b9aa6cde48300787ca", "9bbc027402111984"
C = "fe_bar_lakebase.fraud_dash"
OUT = "/Workspace/Users/alex.owen@databricks.com/FE-BAR-2026-repo/genie/fraud_risk_ops_genie.json"
TOKEN_RX = re.compile(r"dapi[0-9a-f]{20,}|eyJ[A-Za-z0-9_-]{20,}\.|ghp_[A-Za-z0-9]{20,}|password=\S+")

def wh_sql(q):
    r = w.statement_execution.execute_statement(statement=q, warehouse_id=WH, wait_timeout="50s")
    while r.status.state in (StatementState.PENDING, StatementState.RUNNING):
        time.sleep(2); r = w.statement_execution.get_statement(r.statement_id)
    rows = (r.result.data_array or []) if r.result else []
    return r.status.state.value, [c.name for c in r.manifest.schema.columns] if r.manifest else [], rows

# 1. Pick the example transaction: highest-risk block txn with a case, other than the benchmark txn
st, _, rows = wh_sql(f"SELECT txn_id FROM {C}.triage_queue WHERE recommended_action = 'block' AND case_id IS NOT NULL AND txn_id <> 'TXN-0234209' ORDER BY risk_score DESC, txn_id LIMIT 1")
EX_TXN = rows[0][0]
EX_Q = f"Why was {EX_TXN} flagged?"
EX_SQL = (f"SELECT triage_queue.txn_id, triage_queue.risk_score, triage_queue.recommended_action, triage_queue.top_shap_signals "
          f"FROM {C}.triage_queue WHERE triage_queue.txn_id = '{EX_TXN}'")
st, cols, rows = wh_sql(EX_SQL)
print(f"example: {EX_Q}\n  [{st}] cols={cols} rows={rows}")
assert st == "SUCCEEDED" and len(rows) == 1

# 2. Add it to the space (structured surface; nothing else changes)
full = w.genie.get_space(SPACE_ID, include_serialized_space=True)
space = json.loads(full.serialized_space)
exs = space["instructions"]["example_question_sqls"]
print("example SQL before:", len(exs))
if not any(e["question"] == [EX_Q] for e in exs):
    exs.append({"id": uuid.uuid4().hex, "question": [EX_Q], "sql": [EX_SQL],
                "usage_guidance": ["For 'why was <txn_id> flagged' questions, return top_shap_signals as the JSON column; do not split it into separate columns."]})
    exs.sort(key=lambda e: e["id"])
try:
    w.genie.update_space(SPACE_ID, serialized_space=json.dumps(space))
except Exception as e:
    print("update with usage_guidance rejected:", str(e)[:200], "-> retrying without it")
    for e2 in exs: e2.pop("usage_guidance", None)
    w.genie.update_space(SPACE_ID, serialized_space=json.dumps(space))
full = w.genie.get_space(SPACE_ID, include_serialized_space=True)
space = json.loads(full.serialized_space)
print("example SQL after:", len(space["instructions"]["example_question_sqls"]), [e["question"][0] for e in space["instructions"]["example_question_sqls"]])

# 3. Re-export + token scan
export = {"space_id": SPACE_ID, "title": full.title, "description": full.description, "warehouse_id": full.warehouse_id,
          "exported_at": datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"), "exported_in_conversation": "2326365276521178",
          "serialized_space": space}
txt = json.dumps(export, indent=2)
assert not TOKEN_RX.findall(txt)
with open(OUT, "w") as f: f.write(txt + "\n")
print(f"re-exported {OUT} ({len(txt):,} chars); token scan hits: 0")

# 4. Conversation-API benchmarks (same 11 questions and expected values as rounds 0 and 1)
EXPECT = {
  "How much fraud did the model catch vs the baseline rule?": [7010482.03, 933889.13],
  "What is the block-band precision?": [0.5],
  "Why was TXN-0234209 flagged?": [0.98441, "block", "is_new_device", 2.5724, "geo_distance_from_home", 1.8741, "is_online", 1.2001],
  "Which merchant categories have the most block recommendations?": "REFUSE",
  "How many open cases are in each queue?": ["auto_block", 9],
  "How many cases were opened between October 20 and October 23, 2026?": [121],
  "How many cases were created per day and queue in the week starting October 19, 2026?": ["analyst_review", 56, "auto_block", 7, 54, 3],
  "Which 5 customers have the most flagged transactions in the triage queue?": ["CUST-01399", 20, "CUST-00167", 17, "CUST-03895", 16],
  "What is the average monthly spend for customer CUST-08669?": "REFUSE",
  "How many fraud dollars did the model miss, and how many false positives did it generate?": [975725.89, 17630],
  "Which case queue has the most cases with a block recommendation?": ["auto_block", 10],
}
UNAVAILABLE = re.compile(r"not (available|included|present|contain|have|part)|unavailable|doesn.t (have|contain|include)|does not (have|contain|include)|no (data|information|column)|cannot|can.t|isn.t (available|included)|outside", re.I)
def flat(rows):
    out = []
    for r in rows:
        for v in r:
            if v is None: continue
            out.append(str(v)); out += re.findall(r"-?\d+\.?\d*", str(v))
    return out
def has(cells, e):
    if isinstance(e, str): return any(e.lower() in c.lower() for c in cells)
    for c in cells:
        try:
            if abs(float(c) - e) <= 0.01: return True
        except ValueError: pass
    return False
def ask(q):
    t0 = time.time()
    m = w.genie.start_conversation_and_wait(SPACE_ID, q, timeout=datetime.timedelta(seconds=600))
    sql, text, cols, rows = None, "", [], []
    for a in (m.attachments or []):
        if a.text and a.text.content: text += a.text.content + " "
        if a.query and a.query.query:
            sql = a.query.query
            qr = w.genie.get_message_attachment_query_result(SPACE_ID, m.conversation_id, m.message_id, a.attachment_id).statement_response
            cols = [c.name for c in qr.manifest.schema.columns]; rows = (qr.result.data_array or []) if qr.result else []
    return {"status": m.status.value, "conv": m.conversation_id, "msg": m.message_id, "sql": sql, "text": text.strip(), "cols": cols, "rows": rows, "secs": round(time.time() - t0)}
R2 = []
for i, (q, exp) in enumerate(EXPECT.items(), 1):
    r = ask(q)
    ok = (r["status"] == "COMPLETED" and not r["rows"] and bool(UNAVAILABLE.search(r["text"]))) if exp == "REFUSE" else \
         (r["status"] == "COMPLETED" and all(has(flat(r["rows"]), e) for e in exp))
    R2.append((i, q, r, ok))
    print(f"\n#{i} [round2] {q}\n   status={r['status']} secs={r['secs']} conversation={r['conv']} message={r['msg']}")
    print(f"   Genie SQL: {r['sql']}\n   Genie text: {r['text'][:300]}\n   cols={r['cols']} first rows={r['rows'][:5]}\n   => {'PASS' if ok else 'FAIL'}")
# Shape stability for the TXN question: ask it 2 more times
shapes = [len(R2[2][2]["cols"])]
for _ in range(2):
    shapes.append(len(ask("Why was TXN-0234209 flagged?")["cols"]))
print(f"\nTXN-0234209 result column counts over 3 asks: {shapes} (4 = JSON column kept, as in the expected SQL)")
print(f"\nround2_example_sql: Conversation API {sum(ok for *_, ok in R2)}/{len(R2)} PASS")

# 5. Native benchmark eval
run = w.genie.genie_create_eval_run(SPACE_ID).as_dict(); rid = run["eval_run_id"]
for _ in range(90):
    d = w.genie.genie_get_eval_run(SPACE_ID, rid).as_dict()
    if d.get("eval_run_status") in ("DONE", "FAILED", "CANCELLED"): break
    time.sleep(20)
print(f"\nnative eval {rid}: status={d.get('eval_run_status')} correct={d.get('num_correct')} needs_review={d.get('num_needs_review')} done={d.get('num_done')}/{d.get('num_questions')}")
for it in w.genie.genie_list_eval_results(SPACE_ID, rid).as_dict().get("eval_results", []):
    det = w.genie.genie_get_eval_result_details(SPACE_ID, rid, it["result_id"]).as_dict()
    print(f"   {str(det.get('assessment')):13} {it['question']}")
print("\nBefore/after (native GOOD): round0 6/11, round1 8/11, round2", f"{d.get('num_correct')}/{d.get('num_questions')}")

example: Why was TXN-0147293 flagged?
  [SUCCEEDED] cols=['txn_id', 'risk_score', 'recommended_action', 'top_shap_signals'] rows=[['TXN-0147293', '0.9937617359197244', 'block', '[{"feature": "is_new_device", "value": 1.0, "shap": 2.189}, {"feature": "geo_distance_from_home", "value": 160.17, "shap": 1.8438}, {"feature": "risk_segment", "value": "high", "shap": 0.8253}]']]
example SQL before: 4
example SQL after: 5 ['What share of all fraud dollars does each strategy catch?', 'How many cases were opened and closed each week?', 'For open cases, show the top signal and the confidence band', 'Show the top 10 review transactions that do not have a case yet', 'Why was TXN-0147293 flagged?']
re-exported /Workspace/Users/alex.owen@databricks.com/FE-BAR-2026-repo/genie/fraud_risk_ops_genie.json (26,387 chars); token scan hits: 0

#1 [round2] How much fraud did the model catch vs the baseline rule?
   status=COMPLETED secs=32 conversation=01f1d7fc4e631b63880a0bd88170a5af message=01f1d7fc4e781f9c